# Синтетические респонденты CCRS — Llama 3 8B + vLLM

Упрощённая версия исходного `synthetic_respondents_vllm.ipynb`: те же основные функции, один клиент OpenAI, параллельный опрос независимых респондентов. **Режим по умолчанию — `group`**: модель знает о принадлежности респондента к конспирологам, но не получает его реальные значения `ID_Q32` и `ID_Q33`.

Для запуска нужны `marginal.csv` (выборка **уже отобранных** конспирологов), `ccrs_questionnaire.py` и библиотеки `pandas`, `openai`, `tqdm`. Если анкета называется `ccrs_questionnaire(4).py`, переименуй её в `ccrs_questionnaire.py`.

`llama3:8b` — привычное имя из Ollama; в vLLM оно работает **только при настройке алиаса** `--served-model-name llama3:8b`. Без алиаса API обычно использует идентификатор загруженной HF-модели, например `meta-llama/Meta-Llama-3-8B-Instruct`. Перед запуском сверяй идентификатор с `/v1/models` — отдельная ячейка ниже это сделает.

## 1. Импорты и настройки
Меняй в первую очередь `model`, `max_workers`, `n_respondents` и `profile_mode`. Для реального сервера укажи `VLLM_BASE_URL` и при необходимости `VLLM_API_KEY` в переменных окружения либо поменяй значения ниже.

In [49]:
import os
import re
import json
import hashlib
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from ccrs_questionnaire import questionnaire

client = OpenAI(
    base_url=os.getenv('VLLM_BASE_URL', 'http://localhost:8000/v1'),
    api_key=os.getenv('VLLM_API_KEY', 'EMPTY'),
    timeout=120,
    max_retries=2,  # сетевые/серверные ошибки SDK повторяет автоматически
)

model = 'llama3:8b'  # либо ID, который возвращает client.models.list()
profile_mode = 'group'  # 'group', 'individual', 'demography'
temperature = 0.1
max_workers = 8
n_respondents = None  # None -> вся выборка
max_tokens = 8
use_structured_outputs = True  # False, если сервер не поддерживает этот режим
checkpoint_every = 10  # каждые 10 вопросов; 0 -> только после завершения
RUN_GENERATION = True  # True — действительно отправлять запросы на GPU

assert profile_mode in ('group', 'individual', 'demography')

## 2. Данные и профиль — как в исходном ноутбуке

In [27]:
data_path = Path('marginal.csv')
marginal = pd.read_csv(data_path) if data_path.exists() else None
print('Данные:', marginal.shape if marginal is not None else 'marginal.csv не найден')

Данные: (2030, 153)


In [28]:
# Словари декодирования создаём один раз, а не изменяем общий cache из потоков.
DECODERS = {}
for item in questionnaire:
    if 'target' in item and 'options' in item:
        codes = {}
        for label, value in item['options'].items():
            codes.setdefault(value, []).append(label)
        DECODERS[item['target']] = {code: ' / '.join(labels) for code, labels in codes.items()}


def decode_value(row, feature):
    value = row.get(feature)
    if pd.isna(value):
        return 'Unknown'
    return DECODERS.get(feature, {}).get(value, f'Unknown ({value})')


def value_text(value):
    if pd.isna(value):
        return 'Unknown'
    return str(int(value)) if isinstance(value, (int, float)) and float(value).is_integer() else str(value)


def generate_profile(row, profile_mode='group') -> str:
    profile = (
        f'You are {value_text(row.get("ID_Q3"))} years old. '
        f'You live in {decode_value(row, "ID_Q1")}. '
        f'Your gender is {decode_value(row, "ID_Q4")}. '
        f'Your education level is {decode_value(row, "ID_Q5")}. '
        f'Your place of residence is {decode_value(row, "ID_Q6")}. '
        f'Your socioeconomic status is {decode_value(row, "ID_Q8")}. '
        f'Your employment status is {decode_value(row, "ID_Q9")}. '
        f'Politically, you are {value_text(row.get("ID_Q10"))}, where 0 - Left, 10 - Right. '
        f'Your interest in politics is {value_text(row.get("ID_Q11"))}, '
        'where 0 - No interest, 10 - Very interested. '
    )
    if profile_mode in ('group', 'individual'):
        profile += (
            'You belong to a group of people with a stronger-than-average tendency '
            'to believe in conspiracy theories. This does not mean that you believe '
            'every specific conspiracy theory. '
        )
    if profile_mode == 'individual':
        if pd.isna(row.get('ID_Q32')) or pd.isna(row.get('ID_Q33')):
            raise ValueError('В режиме individual обязательны ID_Q32 и ID_Q33')
        profile += (
            f'Your conspiracy belief level is {value_text(row["ID_Q32"])} (0–4). '
            f'Your conspiratorial mentality level is {value_text(row["ID_Q33"])} (0–10). '
        )
    return profile

## 3. Подготовка вопросов
Здесь исправлены две ошибки исходного ноутбука: `enabled=False` и `text_by_country`. Для `{last_national_election}`, `{national_government}`, `{upper_house}` используются *общие формулировки*, не оригинальные страновые тексты CCRS; при необходимости замени их на точные.

In [29]:
COUNTRIES = {
    1: ('Canada', 'English or French', 'Canadian'),
    2: ('United States', 'English', 'American'),
    3: ('Germany', 'German', 'German'),
    4: ('Australia', 'English', 'Australian'),
    5: ('South Africa', 'a South African language', 'South African'),
    6: ('Morocco', 'Arabic or Amazigh', 'Moroccan'),
    7: ('Lebanon', 'Arabic', 'Lebanese'),
    8: ('Brazil', 'Portuguese', 'Brazilian'),
}
PLACEHOLDER = re.compile(r'\{([a-zA-Z_]+)\}')


def prepare_questionnaire(row, items=questionnaire):
    country_code = int(row['ID_Q1'])
    country, language, citizenship = COUNTRIES[country_code]
    values = {
        'country': country, 'language': language, 'citizenship': citizenship,
        'last_national_election': 'the last national election',
        'national_government': 'the national government',
        'upper_house': 'national legislature',
    }
    prepared = []
    for item in items:
        if item['type'] == 'context' or item.get('enabled', True) is False:
            continue
        text = item.get('text', '')
        if 'text_by_country' in item:
            text = item['text_by_country'].get(country_code)
            if text is None:
                continue  # вопрос отсутствует в анкете этой страны
        text = PLACEHOLDER.sub(lambda m: values[m.group(1)], text)
        prepared.append({**item, 'text': text})
    return prepared

## 4. Ответ на один вопрос
Для кодируемых ответов возвращается `int` либо `None` (если в анкете код варианта равен `None`). `free_text` остаётся строкой — её нужно отдельно оценивать по `scorer`. Если сервер поддерживает vLLM Structured Outputs, модель не сможет выбрать код вне допустимого набора.

In [30]:
SYSTEM_PROMPT = (
    'You are participating in an academic sociological survey. '
    'Answer as the person described in your profile, consistently with earlier answers. '
    'Provide only the answer without explanation.'
)


def allowed_codes(item):
    if item['answer_type'] == 'single_choice':
        return list(dict.fromkeys('NA' if v is None else str(v) for v in item['options'].values()))
    if item['answer_type'] == 'integer':
        return [str(n) for n in range(item['min'], item['max'] + 1)]
    return None  # free_text


def answer_the_question(item, context, model, temperature=0.1):
    options = item.get('options')
    if options:
        choices = '\n'.join(f'{"NA" if v is None else v} — {k}' for k, v in options.items())
    else:
        choices = '; '.join(f'{k} = {v}' for k, v in item.get('anchors', {}).items())
    scale = item.get('scale_type', '')
    if 'min' in item:
        scale += f' from {item["min"]} to {item["max"]}'
    history = '\n\n'.join(context)
    prompt = (
        f'Survey history:\n{history}\n\n'
        f'Current question:\n{item["text"]}\n\n'
        f'Scale: {scale}\nAnswer type: {item["answer_type"]}\n'
        f'Answer options:\n{choices}\n\n'
        f'{"Return a short textual answer. If unsure, say I do not know. Do not use NA." if item["answer_type"] == "free_text" else "Return only a valid answer code. Use NA only if it is an allowed option."}'
    )
    codes = allowed_codes(item)
    kwargs = dict(
        model=model, temperature=temperature,
        max_tokens=32 if codes is None else max_tokens,
        messages=[{'role': 'system', 'content': SYSTEM_PROMPT},
                  {'role': 'user', 'content': prompt}],
    )
    if codes and use_structured_outputs:
        kwargs['extra_body'] = {'structured_outputs': {'choice': codes}}

    for attempt in range(3):
        response = client.chat.completions.create(**kwargs)
        raw = (response.choices[0].message.content or '').strip().strip('`"').strip("'")
        if codes is None:
            if raw:
                return raw
        else:
            if raw in codes:
                return None if raw == 'NA' else int(raw)
            # При отключённых structured outputs допускаем точный текст опции.
            if options:
                for label, value in options.items():
                    if raw.casefold() == label.casefold():
                        return value
        if attempt == 2:
            raise ValueError(f'Невалидный ответ {item["target"]}: {raw!r}; допустимо {codes}')
    raise RuntimeError('Не удалось получить ответ')

## 5. Опрос одного респондента
Последовательность вопросов и полная история сохранены из твоей реализации. Каждые `checkpoint_every` вопросов записывается контрольная точка. При повторном запуске опрос продолжается с того же места.

In [31]:
def save_checkpoint(path, answers, context, next_item):
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(
        {'answers': answers, 'context': context, 'next_item': next_item}, ensure_ascii=False
    ), encoding='utf-8')
    tmp.replace(path)


def survey_respondent(profile, questionnaire, model, checkpoint=None, show_progress=False):
    answers = {}
    context = [f'Profile: {profile}']
    start = 0
    if checkpoint and checkpoint.exists():
        saved = json.loads(checkpoint.read_text(encoding='utf-8'))
        answers, context, start = saved['answers'], saved['context'], saved['next_item']

    for i in tqdm(range(start, len(questionnaire)), desc='Survey progress', disable=not show_progress):
        item = questionnaire[i]
        if item['type'] in ('heading', 'instruction'):
            context.append(item['text'])
        elif item['type'] == 'question':
            answer = answer_the_question(item, context, model, temperature=temperature)
            answers[item['target']] = answer
            context.append(f'Question: {item["text"]}\nAnswer: {answer if answer is not None else "NA"}')
        # context уже исключён в prepare_questionnaire()
        if checkpoint and checkpoint_every and (i + 1) % checkpoint_every == 0:
            save_checkpoint(checkpoint, answers, context, i + 1)

    if checkpoint:
        save_checkpoint(checkpoint, answers, context, len(questionnaire))
    return answers, context

## 6. Несколько респондентов параллельно
Как в исходном ноутбуке, параллелим **респондентов**, а вопросы одного респондента идут последовательно. `as_completed()` показывает выполнение по факту завершения. Контрольные точки и результаты хранятся в отдельных файлах, поэтому один сбой не уничтожает остальные опросы.

In [32]:
def survey_respondents(data, questionnaire, model, max_workers=16, profile_mode='group'):
    settings = {
        'model': model, 'profile_mode': profile_mode, 'temperature': temperature,
        'max_tokens': max_tokens, 'structured': use_structured_outputs,
        'system_prompt': SYSTEM_PROMPT, 'questionnaire': questionnaire,
        'countries': COUNTRIES, 'checkpoint_every': checkpoint_every,
    }
    run_id = hashlib.sha256(json.dumps(settings, sort_keys=True, default=str).encode()).hexdigest()[:12]
    output_dir = Path('results') / run_id
    output_dir.mkdir(parents=True, exist_ok=True)
    rows = list(data.iterrows())
    answers, contexts = [None] * len(rows), [None] * len(rows)
    errors = {}

    def survey_row(position, row):
        profile = generate_profile(row, profile_mode)
        items = prepare_questionnaire(row, questionnaire)
        # Отпечаток учитывает всю исходную строку: изменили респондента -> новый checkpoint.
        row_id = hashlib.sha256(json.dumps(row.to_dict(), sort_keys=True, default=str).encode()).hexdigest()[:12]
        checkpoint = output_dir / f'row_{position}_{row_id}.json'
        return survey_respondent(profile, items, model, checkpoint=checkpoint)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(survey_row, i, row): i for i, (_, row) in enumerate(rows)}
        for future in tqdm(as_completed(futures), total=len(futures), desc='Respondents'):
            i = futures[future]
            try:
                answers[i], contexts[i] = future.result()
            except Exception as exc:
                errors[i] = f'{type(exc).__name__}: {exc}'
                tqdm.write(f'Респондент {i}: {errors[i]}')

    result = pd.DataFrame([answer or {} for answer in answers])
    result.insert(0, 'source_index', [idx for idx, _ in rows])
    result.to_csv(output_dir / 'answers.csv', index=False)
    if errors:
        (output_dir / 'errors.json').write_text(json.dumps(errors, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Сохранено: {output_dir / "answers.csv"}; ошибок: {len(errors)}')
    return answers, contexts

## 7. Проверка модели и запуск
`client.models.list()` требует доступного vLLM-сервера, поэтому по умолчанию запросы **выключены**. На immers.cloud сначала настрой адрес и модель, затем поставь `RUN_GENERATION=True`. Если в `/v1/models` другое имя Llama 3 8B — скопируй его в `model` выше.

In [50]:
if marginal is not None:
    row = marginal.iloc[0]
    print('Профиль:', generate_profile(row, profile_mode))
    print('Вопросов для страны:', sum(x['type'] == 'question' for x in prepare_questionnaire(row)))

if RUN_GENERATION:
    if marginal is None:
        raise FileNotFoundError('Не найден marginal.csv')
    available = [x.id for x in client.models.list().data]
    print('Модели на сервере:', available)
    if model not in available:
        raise ValueError(f'Модель {model!r} недоступна. Используй имя из списка выше.')
    sample = marginal if n_respondents is None else marginal.iloc[:n_respondents]
    started = time.perf_counter()
    answers, contexts = survey_respondents(
        sample, questionnaire, model, max_workers=max_workers, profile_mode=profile_mode
    )
    ok = sum(answer is not None for answer in answers)
    elapsed = time.perf_counter() - started
    print(f'{ok}/{len(answers)} респондентов за {elapsed:.1f} с; '
          f'{60 * ok / elapsed:.2f} респондентов/мин' if elapsed else 'Нет результатов')

Профиль: You are 19 years old. You live in Australia. Your gender is Male. Your education level is Unknown (-1.0). Your place of residence is A suburb or outskirt of a big city. Your socioeconomic status is Middle class. Your employment status is Working for pay full-time. Politically, you are 5, where 0 - Left, 10 - Right. Your interest in politics is 0, where 0 - No interest, 10 - Very interested. You belong to a group of people with a stronger-than-average tendency to believe in conspiracy theories. This does not mean that you believe every specific conspiracy theory. 
Вопросов для страны: 137
Модели на сервере: ['llama3:8b']


Respondents: 100%|██████████| 2030/2030 [1:14:39<00:00,  2.21s/it]

Сохранено: results/7492f40b7a05/answers.csv; ошибок: 0
2030/2030 респондентов за 4480.2 с; 27.19 респондентов/мин


## Примечания
- `group`: явно указывает принадлежность к конспирологам, без значений `ID_Q32`, `ID_Q33`. `individual`: дополнительно передаёт эти два показателя (есть утечка информации о целевых ответах). `demography`: только базовый профиль, включая политические характеристики.
- `marginal.csv` здесь **не создаётся**: он должен содержать нужную целевую группу.
- При повторном запуске готовые респонденты считываются из файлов, частично готовые продолжаются с контрольной точки. Не удаляй папку `results`, если нужен перезапуск.
- `free_text` сохраняется как текст, числовые переменные — как коды. Преобразование CRT-ответов в баллы остаётся отдельной задачей.
- Для настоящего воспроизведения CCRS замени общие шаблоны национальных выборов и институтов на страновые формулировки.
- Для версии **Llama 3 8B** (не 3.1) исходный контекст обычно ограничен 8192 токенами. При очень длинных историях контролируй предел контекста на сервере.